# Hướng dẫn chạy Offline Pipeline trên Kaggle (TransNetV2 + FAISS)

Notebook này được thiết kế để tận dụng GPU miễn phí của Kaggle nhằm trích xuất keyframe phân cảnh thông minh bằng TransNetV2, OCR (EasyOCR), YOLO và tạo chỉ mục FAISS siêu tốc.

### Chuẩn bị:
1. Bật **GPU Accelerator (T4 x2)** trong phần Settings của Kaggle.
2. Upload video của bạn vào phần **Input** (tạo Dataset mới).
3. Chạy lần lượt các ô code dưới đây.

In [ ]:
# 1. Reset working directory & Tải source code mới nhất
%cd /kaggle/working
!rm -rf /kaggle/working/agent_retrieval
!git clone https://github.com/Thien-Dan/agent_retrieval.git
%cd /kaggle/working/agent_retrieval/video_retrieval

# 2. Cài đặt toàn bộ dependencies
!pip install -r requirements.txt
!pip install loguru rank-bm25 faiss-cpu ultralytics easyocr einops

# Fix meta tensors bug on Kaggle
!pip uninstall -y accelerate
!pip install transformers==4.37.2


In [ ]:
%cd /kaggle/working/agent_retrieval/video_retrieval
# 3. Dọn dẹp dữ liệu cũ (nếu có) để tránh xung đột
!rm -rf data/indexes/*
!rm -rf data/frames/*

# 4. Cấu hình tự động dùng EasyOCR và chế độ trích xuất TransNetV2
import re

ocr_config = "config/ocr_config.py"
with open(ocr_config, "r", encoding="utf-8") as f:
    code = f.read()
code = re.sub(r'engine: str = Field\(\s*default="paddle"', 'engine: str = Field(\n        default="easyocr"', code)
with open(ocr_config, "w", encoding="utf-8") as f:
    f.write(code)

# Thiết lập chế độ trích xuất Keyframe thông minh TransNetV2
frame_config = "config/frame_config.py"
with open(frame_config, "r", encoding="utf-8") as f:
    code = f.read()
code = re.sub(r'mode: ExtractionMode = Field\(\s*default=ExtractionMode\.\w+', 'mode: ExtractionMode = Field(\n        default=ExtractionMode.TRANSNETV2', code)
with open(frame_config, "w", encoding="utf-8") as f:
    f.write(code)

print("✅ Cấu hình hoàn tất: Dùng EasyOCR + TransNetV2 Keyframe Extraction")

In [ ]:
%cd /kaggle/working/agent_retrieval/video_retrieval
# 5. Chạy quá trình trích xuất bằng script Kaggle Pipeline
!python scripts/kaggle_pipeline.py

In [ ]:
%cd /kaggle/working/agent_retrieval/video_retrieval
# 6. Đóng gói thư mục data/indexes và data/frames thành 2 file ZIP riêng biệt để download
import shutil
import os
print("Đang nén thư mục data/indexes (chứa SQLite, FAISS Index, BM25)...")
shutil.make_archive("/kaggle/working/kaggle_indexes", "zip", "data/indexes")
print("✅ Hoàn tất nén data/indexes!")
if os.path.exists("data/frames"):
    print("\nĐang nén thư mục data/frames (chứa ảnh, file có thể rất nặng)...")
    shutil.make_archive("/kaggle/working/kaggle_frames", "zip", "data/frames")
    print("✅ Hoàn tất nén data/frames!")
print("\n✅ Đã xong! Hãy tải kaggle_indexes.zip và kaggle_frames.zip ở cột bên phải về máy tính.")
